# 05 — Evaluation & Interpretability

1. Load trained models and run full LOSO evaluation
2. Confusion matrices for all models
3. Comparison table (accuracy, macro-F1, balanced accuracy)
4. LIME explanations for GNN+LSTM predictions
5. Model latency profiling

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time
import json
from pathlib import Path

from src.config import PROCESSED_DIR, METRICS_DIR, MODELS_DIR, PAMAP2_ACTIVITIES
from src.evaluation import compute_metrics, plot_confusion_matrix, get_predictions
from src.train import get_device

sns.set_theme(style='whitegrid')
DEVICE = get_device()
print(f'Device: {DEVICE}')

## 1. Load Data & Saved Predictions

In [ ]:
try:
    X   = np.load(f'{PROCESSED_DIR}/pamap2_X.npy')
    y   = np.load(f'{PROCESSED_DIR}/pamap2_y.npy')
    subj = np.load(f'{PROCESSED_DIR}/pamap2_subjects.npy')
    N_CLASSES = len(np.unique(y))
    activity_names = [name for name in sorted(PAMAP2_ACTIVITIES.values())][:N_CLASSES]
    print(f'Loaded: {X.shape}, {N_CLASSES} classes')
    DATA_LOADED = True
except FileNotFoundError:
    print('Run preprocessing and training notebooks first.')
    DATA_LOADED = False

## 2. Load Saved Predictions & Compute Metrics

In [ ]:
metrics_dir = Path(METRICS_DIR)

# Collect all saved prediction pairs
pred_files = sorted(metrics_dir.glob('*_y_pred.npy'))
summary_rows = []

for pred_file in pred_files:
    run_name = pred_file.stem.replace('_y_pred', '')
    true_file = metrics_dir / f'{run_name}_y_true.npy'
    if not true_file.exists():
        continue

    y_true = np.load(true_file)
    y_pred = np.load(pred_file)
    m = compute_metrics(y_true, y_pred)

    summary_rows.append({
        'Run': run_name,
        'Accuracy': f"{m['accuracy']:.4f}",
        'Balanced Acc': f"{m['balanced_accuracy']:.4f}",
        'Macro F1': f"{m['macro_f1']:.4f}",
    })

if summary_rows:
    print(pd.DataFrame(summary_rows).set_index('Run').to_string())
else:
    print('No prediction files found yet — run training notebooks first.')

## 3. Confusion Matrices

In [ ]:
for pred_file in pred_files:
    run_name = pred_file.stem.replace('_y_pred', '')
    true_file = metrics_dir / f'{run_name}_y_true.npy'
    if not true_file.exists():
        continue

    y_true = np.load(true_file)
    y_pred = np.load(pred_file)
    m = compute_metrics(y_true, y_pred)

    plot_confusion_matrix(
        m['confusion_matrix'],
        label_names=activity_names if DATA_LOADED else None,
        title=f'Confusion Matrix — {run_name}',
        save_name=f'cm_{run_name}',
    )

## 4. All-Model Comparison Chart

In [ ]:
# Merge baseline JSON + deep model predictions
baseline_file = metrics_dir / 'pamap2_baselines.json'
all_results = {}

if baseline_file.exists():
    with open(baseline_file) as f:
        baselines = json.load(f)
    for k, v in baselines.items():
        all_results[k] = {'mean_accuracy': v['mean_accuracy'], 'std': v['std_accuracy']}

for pred_file in pred_files:
    run_name = pred_file.stem.replace('_y_pred', '')
    true_file = metrics_dir / f'{run_name}_y_true.npy'
    if not true_file.exists():
        continue
    y_true = np.load(true_file)
    y_pred = np.load(pred_file)
    m = compute_metrics(y_true, y_pred)
    all_results[run_name] = {'mean_accuracy': m['accuracy'], 'std': 0.0}

if all_results:
    fig, ax = plt.subplots(figsize=(10, 5))
    names  = list(all_results.keys())
    accs   = [all_results[n]['mean_accuracy'] for n in names]
    errs   = [all_results[n]['std'] for n in names]
    colors = plt.cm.Set2(np.linspace(0, 1, len(names)))
    ax.barh(names, accs, xerr=errs, color=colors, capsize=4)
    ax.set_xlim(0, 1)
    ax.set_xlabel('LOSO Accuracy')
    ax.set_title('Model Comparison — PAMAP2 LOSO Evaluation')
    plt.tight_layout()
    plt.show()
else:
    print('No results to plot yet.')

## 5. LIME Explanations

In [ ]:
# LIME over engineered features (model-agnostic)
if DATA_LOADED:
    import lime
    import lime.lime_tabular
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import StandardScaler
    from src.baselines import extract_features
    from src.config import SEED

    X_feat = extract_features(X)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_feat)

    rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
    rf.fit(X_scaled, y)

    n_channels = X.shape[2]
    feature_names = (
        [f'mean_ch{i}' for i in range(n_channels)] +
        [f'std_ch{i}'  for i in range(n_channels)] +
        [f'min_ch{i}'  for i in range(n_channels)] +
        [f'max_ch{i}'  for i in range(n_channels)] +
        [f'rms_ch{i}'  for i in range(n_channels)] +
        [f'fft_ch{i}'  for i in range(n_channels)]
    )

    explainer = lime.lime_tabular.LimeTabularExplainer(
        X_scaled,
        feature_names=feature_names,
        class_names=activity_names,
        mode='classification',
        random_state=SEED,
    )

    # Explain one instance
    idx = 42
    exp = explainer.explain_instance(
        X_scaled[idx],
        rf.predict_proba,
        num_features=15,
        top_labels=1,
    )
    exp.as_pyplot_figure()
    plt.title(f'LIME Explanation — Instance {idx} (true label: {activity_names[y[idx]]})')
    plt.tight_layout()
    plt.show()
else:
    print('Data not loaded.')

## 6. Model Latency Profiling

In [ ]:
if DATA_LOADED:
    from src.models import GNNLSTMModel, LSTMOnlyModel, GNNOnlyModel
    from src.dataset import HARSequenceDataset, HARWindowDataset, HARGraphDataset
    from src.graph_construction import build_pamap2_adj

    N_NODES = 3
    seq_ds = HARSequenceDataset(X, y, dataset='pamap2', seq_len=10)
    x_sample, adj_sample, _ = seq_ds[0]
    NODE_FEAT = x_sample.shape[2]
    FLAT_DIM  = X.shape[1] * X.shape[2]

    models_to_profile = {
        'LSTM-only': (LSTMOnlyModel(FLAT_DIM, N_CLASSES), False),
        'GNN-only':  (GNNOnlyModel(NODE_FEAT, N_NODES, N_CLASSES), True),
        'GNN+LSTM':  (GNNLSTMModel(NODE_FEAT, N_NODES, N_CLASSES), True),
    }

    adj = build_pamap2_adj()
    batch_size = 1  # single-sample inference (mobile scenario)
    n_reps = 100

    latency_rows = []
    for name, (model, use_adj) in models_to_profile.items():
        model.eval()
        with torch.no_grad():
            if use_adj and name == 'GNN+LSTM':
                dummy = x_sample.unsqueeze(0)  # (1, seq_len, n_nodes, feat)
                dummy_adj = adj
                # warm-up
                _ = model(dummy, dummy_adj)
                t0 = time.perf_counter()
                for _ in range(n_reps):
                    model(dummy, dummy_adj)
            elif use_adj:
                graph_ds = HARGraphDataset(X[:1], y[:1], dataset='pamap2')
                dummy, dummy_adj, _ = graph_ds[0]
                dummy = dummy.unsqueeze(0)
                _ = model(dummy, dummy_adj)
                t0 = time.perf_counter()
                for _ in range(n_reps):
                    model(dummy, dummy_adj)
            else:
                flat_ds = HARWindowDataset(X[:1], y[:1])
                dummy, _ = flat_ds[0]
                dummy = dummy.unsqueeze(0).unsqueeze(0)  # (1, 1, flat_dim)
                _ = model(dummy)
                t0 = time.perf_counter()
                for _ in range(n_reps):
                    model(dummy)

            elapsed_ms = (time.perf_counter() - t0) / n_reps * 1000
            n_params = sum(p.numel() for p in model.parameters())
            latency_rows.append({
                'Model': name,
                'Params': f'{n_params:,}',
                'Latency (ms/sample)': f'{elapsed_ms:.3f}',
            })
            print(f'{name:12s}: {n_params:>8,} params  |  {elapsed_ms:.3f} ms/sample')

    pd.DataFrame(latency_rows).set_index('Model')